# Session 6: Tool Use, Function Calling, and ReAct
### Agentic AI Nano Bootcamp | Day 2, Session 6

---

## Learning Objectives

By the end of this session, you will be able to:
- Define and register tools using the OpenAI function-calling schema
- Build agents that select and invoke tools based on their structured definitions
- Implement the ReAct (Reason + Act) pattern for transparent, traceable reasoning
- Manage agentic memory: in-context, key-value, and vector-based
- Describe the principles of multi-agent collaboration

## Session Outline

1. OpenAI Function Calling
2. Tool Schema Design
3. The ReAct Pattern
4. Agentic Memory Management
5. Multi-Agent Collaboration — Concepts
6. Lab: Tool-Using Agent with Function Calling
7. Lab: ReAct Agent with Full Trace
8. Lab: Memory-Persistent Agent

---

In [ ]:
import subprocess, os, json, math, datetime
subprocess.run(['pip', 'install', 'openai', 'python-dotenv', '-q'], capture_output=True)

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from typing import Any

client = OpenAI()
print("Environment ready.")

## 1. OpenAI Function Calling

**Function calling** (also called tool use) is a feature of the OpenAI chat API that allows the model to request structured invocation of external functions rather than generating free text.

### How It Works

```
1. Developer provides tool schemas in the API request.

2. When the LLM determines a tool is needed, it responds with:
     finish_reason = "tool_calls"
     tool_calls    = [{"name": "get_weather", "arguments": "{\"city\": \"Chennai\"}"}]

3. Developer executes the function and adds the result to the messages array.

4. The LLM receives the result and continues reasoning.
```

### Why Use Native Function Calling vs. Prompt-Based?

| Dimension | Prompt-based (Session 5) | Native Function Calling |
|---|---|---|
| Schema enforcement | Manual JSON parsing | Enforced by API |
| Reliability | Prone to formatting errors | Structured, validated |
| Multi-tool selection | One at a time | Parallel calls supported |
| Debugging | Parse errors common | Always valid JSON |
| Production suitability | Prototyping | Production-grade |

## 2. Tool Schema Design

Each tool is described in JSON Schema format. The model reads these descriptions to decide when and how to call a tool.

### Schema Structure

```json
{
  "type": "function",
  "function": {
    "name": "get_network_status",
    "description": "Retrieve the current status of a network node by its ID. Call this when the user asks about outages, connectivity, or node health.",
    "parameters": {
      "type": "object",
      "properties": {
        "node_id": {
          "type": "string",
          "description": "The unique identifier of the network node, e.g. 'NODE-TN-0042'."
        },
        "include_history": {
          "type": "boolean",
          "description": "If true, include the last 24 hours of status events."
        }
      },
      "required": ["node_id"]
    }
  }
}
```

### Schema Writing Best Practices

1. **Name**: use snake_case, be specific. `get_subscriber_churn_rate` not `get_data`.
2. **Description**: explain *when* to call it, not just *what* it does. The model reads this to decide.
3. **Parameters**: describe each field with type and purpose. The model uses these to fill in arguments.
4. **Required fields**: mark only truly mandatory parameters as required.
5. **Enums**: use `enum` to restrict string parameters to valid values.

In [ ]:
# Define tools using OpenAI's function-calling schema
# Each entry has a "schema" for the API and an "implementation" function

import random

# --- Tool implementations ---

def get_network_node_status(node_id: str, include_history: bool = False) -> dict:
    """Simulate network node status lookup."""
    statuses = ["operational", "degraded", "offline", "maintenance"]
    random.seed(hash(node_id) % 1000)
    status = random.choice(statuses)
    result = {
        "node_id":      node_id,
        "status":       status,
        "uptime_pct":   round(random.uniform(95.0, 99.9), 2),
        "active_alarms": random.randint(0, 5),
        "last_updated": datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    }
    if include_history:
        result["history"] = [
            {"time": "06:00", "event": "Threshold alert cleared"},
            {"time": "02:30", "event": "Traffic spike +40%"},
        ]
    return result

def get_subscriber_stats(area_code: str, metric: str = "all") -> dict:
    """Retrieve subscriber statistics for a geographic area."""
    random.seed(hash(area_code) % 999)
    data = {
        "area_code":          area_code,
        "total_subscribers":  random.randint(50000, 500000),
        "active_today":       random.randint(30000, 450000),
        "avg_data_usage_gb":  round(random.uniform(4.0, 18.0), 1),
        "churn_rate_pct":     round(random.uniform(1.2, 4.5), 2),
        "plan_distribution":  {"basic": 35, "standard": 45, "premium": 20},
    }
    if metric != "all" and metric in data:
        return {metric: data[metric]}
    return data

def calculate_revenue_impact(subscribers: int, churn_rate_pct: float,
                              avg_monthly_revenue: float) -> dict:
    """Calculate the monthly revenue impact of subscriber churn."""
    churned     = int(subscribers * churn_rate_pct / 100)
    lost_revenue = round(churned * avg_monthly_revenue, 2)
    return {
        "subscribers_at_risk":  churned,
        "monthly_revenue_loss": lost_revenue,
        "annual_revenue_loss":  round(lost_revenue * 12, 2),
    }

def create_incident_ticket(title: str, priority: str,
                            description: str, assigned_team: str) -> dict:
    """Create a network incident ticket in the ticketing system."""
    ticket_id = f"INC-{random.randint(10000,99999)}"
    return {
        "ticket_id":     ticket_id,
        "title":         title,
        "priority":      priority,
        "status":        "open",
        "assigned_team": assigned_team,
        "created_at":    datetime.datetime.now().isoformat(),
        "message":       f"Ticket {ticket_id} created and assigned to {assigned_team}.",
    }

# --- Tool schemas (JSON Schema format for OpenAI API) ---

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_network_node_status",
            "description": "Retrieve the current operational status, uptime, and alarm count for a network node. Call this when asked about node health, outages, or connectivity issues.",
            "parameters": {
                "type": "object",
                "properties": {
                    "node_id":         {"type": "string",  "description": "Network node identifier, e.g. 'NODE-TN-0042'"},
                    "include_history": {"type": "boolean", "description": "Set true to include last 24h event history"}
                },
                "required": ["node_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_subscriber_stats",
            "description": "Retrieve subscriber statistics for a geographic area. Use this for questions about subscriber count, churn, data usage, or plan distribution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "area_code": {"type": "string", "description": "Geographic area code, e.g. 'TN-CHN' for Chennai Tamil Nadu"},
                    "metric":    {"type": "string", "description": "Specific metric to retrieve. Use 'all' for full data.",
                                  "enum": ["all", "total_subscribers", "churn_rate_pct", "avg_data_usage_gb", "plan_distribution"]}
                },
                "required": ["area_code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_revenue_impact",
            "description": "Calculate monthly and annual revenue loss due to subscriber churn. Call this after retrieving subscriber stats when revenue analysis is needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "subscribers":          {"type": "integer", "description": "Total subscriber count"},
                    "churn_rate_pct":       {"type": "number",  "description": "Monthly churn rate as a percentage"},
                    "avg_monthly_revenue":  {"type": "number",  "description": "Average monthly revenue per subscriber in rupees"}
                },
                "required": ["subscribers", "churn_rate_pct", "avg_monthly_revenue"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_incident_ticket",
            "description": "Create an incident ticket in the NOC ticketing system. Call this when an issue requires escalation or tracking.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title":         {"type": "string", "description": "Short descriptive title for the incident"},
                    "priority":      {"type": "string", "description": "Ticket priority", "enum": ["P1", "P2", "P3", "P4"]},
                    "description":   {"type": "string", "description": "Detailed description of the incident"},
                    "assigned_team": {"type": "string", "description": "Team responsible, e.g. 'NOC-L2', 'Field-Engineering'",
                                      "enum": ["NOC-L1", "NOC-L2", "Field-Engineering", "Billing-Ops", "Network-Planning"]}
                },
                "required": ["title", "priority", "description", "assigned_team"]
            }
        }
    }
]

# Map name -> implementation
TOOL_IMPL = {
    "get_network_node_status":   get_network_node_status,
    "get_subscriber_stats":      get_subscriber_stats,
    "calculate_revenue_impact":  calculate_revenue_impact,
    "create_incident_ticket":    create_incident_ticket,
}

print(f"Registered {len(TOOLS)} tools:")
for t in TOOLS:
    print(f"  - {t['function']['name']}")

## 3. The ReAct Pattern

**ReAct** (Yao et al., 2022) interleaves *reasoning traces* with *actions*. Instead of acting immediately, the agent first writes an explicit thought, then acts, then observes, then reasons again.

### ReAct vs. Plain Action

```
Plain (no reasoning trace):
  User query  ->  [tool call]  ->  [tool call]  ->  Answer

ReAct:
  User query
    Thought: I need to find the node status first...
    Action:  get_network_node_status(node_id="NODE-TN-042")
    Obs:     {status: "degraded", alarms: 3}
    Thought: Node is degraded with 3 alarms. I should create a ticket...
    Action:  create_incident_ticket(title="NODE-TN-042 degraded", ...)
    Obs:     {ticket_id: "INC-84921"}
    Thought: Ticket created. I have enough information to answer.
    Answer:  Node is degraded. Ticket INC-84921 raised for NOC-L2.
```

### Why ReAct Improves Reliability

- The thought step forces the model to commit to a reasoning path before acting, reducing impulsive or incorrect tool selection.
- The trace is inspectable — engineers can debug exactly why the agent took each step.
- Observations are explicitly acknowledged, so the agent cannot ignore tool results.

## Lab 1: Tool-Using Agent with OpenAI Function Calling

We build an agent that uses the native OpenAI function-calling API to reliably select and execute tools across multiple turns.

In [ ]:
class FunctionCallingAgent:
    """
    Agent using OpenAI native function calling.
    The model returns structured tool_calls; we execute them and feed results back.
    """

    SYSTEM = """
You are a senior NOC (Network Operations Centre) analyst AI assistant.
You have access to tools for querying network status, subscriber data,
revenue calculations, and incident management.

Approach every task methodically:
1. Gather all necessary data before drawing conclusions.
2. Do not make assumptions about numbers — always use tools to retrieve real values.
3. When creating tickets, use the appropriate priority and team based on the severity.
4. Produce a clear, structured final response citing the data you retrieved.
"""

    def __init__(self, tools: list, tool_impl: dict,
                 model: str = "gpt-4o-mini", max_turns: int = 10, verbose: bool = True):
        self.tools     = tools
        self.tool_impl = tool_impl
        self.model     = model
        self.max_turns = max_turns
        self.verbose   = verbose

    def _execute_tool(self, name: str, arguments: str) -> str:
        try:
            args = json.loads(arguments)
            result = self.tool_impl[name](**args)
            return json.dumps(result, indent=2)
        except Exception as e:
            return json.dumps({"error": str(e)})

    def run(self, user_request: str) -> str:
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user",   "content": user_request}
        ]

        if self.verbose:
            print("=" * 65)
            print(f"  REQUEST: {user_request}")
            print("=" * 65)

        for turn in range(self.max_turns):
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=self.tools,
                tool_choice="auto",
                temperature=0.1,
                max_tokens=1000
            )

            msg     = response.choices[0].message
            reason  = response.choices[0].finish_reason

            messages.append(msg)

            if reason == "stop":
                if self.verbose:
                    print(f"\n  [Turn {turn+1}] FINISHED")
                return msg.content

            if reason == "tool_calls":
                for tc in msg.tool_calls:
                    fn_name = tc.function.name
                    fn_args = tc.function.arguments

                    if self.verbose:
                        print(f"\n  [Turn {turn+1}] TOOL CALL: {fn_name}")
                        try:
                            parsed_args = json.loads(fn_args)
                            for k, v in parsed_args.items():
                                print(f"    {k}: {v}")
                        except Exception:
                            print(f"    args: {fn_args}")

                    result = self._execute_tool(fn_name, fn_args)

                    if self.verbose:
                        print(f"  [Turn {turn+1}] RESULT:")
                        for line in result.splitlines()[:8]:
                            print(f"    {line}")
                        if len(result.splitlines()) > 8:
                            print(f"    ... ({len(result.splitlines())} lines total)")

                    messages.append({
                        "role":         "tool",
                        "tool_call_id": tc.id,
                        "name":         fn_name,
                        "content":      result
                    })

        return "Agent exceeded maximum turns."


noc_agent = FunctionCallingAgent(tools=TOOLS, tool_impl=TOOL_IMPL, verbose=True)
print("FunctionCallingAgent ready.")

In [ ]:
# Task 1: Multi-tool analysis

answer = noc_agent.run(
    "Check the status of network node NODE-TN-0042 including its history. "
    "If it has any active alarms or is not fully operational, "
    "create a P2 ticket assigned to NOC-L2 and include the node's uptime in the ticket description."
)

print("\n" + "=" * 65)
print("  FINAL ANSWER")
print("=" * 65)
print(answer)

In [ ]:
# Task 2: Revenue impact analysis

answer = noc_agent.run(
    "Get subscriber statistics for area TN-CHN (Chennai). "
    "Then calculate the monthly and annual revenue impact of churn, "
    "assuming an average monthly revenue of 450 rupees per subscriber. "
    "Present the findings as a brief management summary with a recommendation."
)

print("\n" + "=" * 65)
print("  FINAL ANSWER")
print("=" * 65)
print(answer)

## Lab 2: ReAct Agent with Explicit Reasoning Trace

We build an agent that externalises its reasoning — every thought is captured, logged, and surfaced for inspection. This is the ReAct pattern in its clearest form.

In [ ]:
class ReActAgent:
    """
    ReAct agent: Reason + Act interleaved.
    Every step is: Thought -> Action -> Observation -> Thought -> ...
    The full trace is stored and returned alongside the answer.
    """

    SYSTEM = """
You are a precise telecom network analyst using the ReAct reasoning pattern.

At every step you MUST produce a JSON object with exactly these fields:
{
  "thought": "your detailed reasoning about the current situation and what to do next",
  "action": "tool_name OR 'finish'",
  "action_input": {the arguments object for the tool, OR the final answer string if action=finish}
}

Rules:
- Never skip the thought field. Reasoning must precede every action.
- After each observation, reassess: do you have enough information to finish?
- If action is 'finish', action_input must be a comprehensive answer string.
- Use tools sequentially — retrieve data before computing, compute before reporting.
"""

    def __init__(self, tools: list, tool_impl: dict,
                 model: str = "gpt-4o-mini", max_steps: int = 10):
        self.tools     = tools
        self.tool_impl = tool_impl
        self.model     = model
        self.max_steps = max_steps
        self.trace     = []

    def _execute(self, tool_name: str, tool_input: Any) -> str:
        if tool_name not in self.tool_impl:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        try:
            if isinstance(tool_input, dict):
                result = self.tool_impl[tool_name](**tool_input)
            else:
                result = self.tool_impl[tool_name](tool_input)
            return json.dumps(result, indent=2)
        except Exception as e:
            return json.dumps({"error": str(e)})

    def run(self, task: str) -> dict:
        self.trace = []
        messages   = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user",   "content": f"Task: {task}"}
        ]

        print("=" * 65)
        print(f"  TASK: {task}")
        print("=" * 65)

        for step in range(1, self.max_steps + 1):
            raw = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.1,
                max_tokens=800,
                response_format={"type": "json_object"}
            ).choices[0].message.content

            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                print(f"\n  [Step {step}] JSON parse error. Stopping.")
                break

            thought    = parsed.get("thought", "")
            action     = parsed.get("action", "")
            action_in  = parsed.get("action_input", {})

            print(f"\n  [Step {step}] THOUGHT")
            print(f"  {thought}")

            self.trace.append({"step": step, "thought": thought, "action": action})

            if action.lower() == "finish":
                answer = action_in if isinstance(action_in, str) else json.dumps(action_in)
                print(f"\n  [Step {step}] FINISH")
                self.trace[-1]["observation"] = answer
                return {"answer": answer, "trace": self.trace, "steps": step}

            print(f"  [Step {step}] ACTION: {action}")
            if isinstance(action_in, dict):
                for k, v in action_in.items():
                    print(f"    {k}: {v}")

            observation = self._execute(action, action_in)
            print(f"  [Step {step}] OBSERVATION:")
            for line in observation.splitlines()[:6]:
                print(f"    {line}")

            self.trace[-1]["observation"] = observation

            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user",      "content": f"Observation from {action}: {observation}"})

        return {"answer": "Max steps reached.", "trace": self.trace, "steps": self.max_steps}


react_agent = ReActAgent(tools=TOOLS, tool_impl=TOOL_IMPL, max_steps=8)
print("ReActAgent ready.")

In [ ]:
# Run ReAct agent and inspect the reasoning trace

result = react_agent.run(
    "Analyse area MH-MUM (Mumbai). Get full subscriber stats. "
    "Calculate churn revenue impact assuming 600 rupees average monthly revenue. "
    "If monthly revenue loss exceeds 10 million rupees, raise a P1 ticket "
    "assigned to Network-Planning with a description citing the numbers. "
    "Present a structured summary."
)

print("\n" + "=" * 65)
print("  FINAL ANSWER")
print("=" * 65)
print(result["answer"])

print("\n" + "=" * 65)
print(f"  TRACE SUMMARY: {result['steps']} steps")
print("=" * 65)
for entry in result["trace"]:
    print(f"  Step {entry['step']:>2} | Action: {entry['action']:<35} | Thought excerpt: {str(entry['thought'])[:50]}...")

## 4. Agentic Memory Management

Memory is what separates a stateful agent from a stateless one. Agents need different types of memory depending on the time horizon of the information.

### Memory Taxonomy

| Type | Scope | Implementation | Use case |
|---|---|---|---|
| **Sensory** | Current turn only | Input tokens | The current user message |
| **Working (in-context)** | Current session | Messages array | Conversation history, scratchpad |
| **Episodic (short-term)** | Across sessions | Key-value store | User preferences, recent actions |
| **Semantic (long-term)** | Permanent | Vector DB | Domain knowledge, past cases |
| **Procedural** | Encoded in weights | Model fine-tuning | Skills, domain expertise |

### The Context Window Constraint

All in-context memory is bounded by the model's context window (e.g. 128K tokens for GPT-4o). For long-running agents, strategies include:
- **Summarisation**: periodically compress the conversation history
- **Sliding window**: keep the most recent N turns only
- **External memory**: write key facts to a store and retrieve them on demand

## Lab 3: Memory-Persistent Agent

We add an external key-value memory store to the agent so it can remember facts across separate invocations.

In [ ]:
class AgentMemory:
    """
    A simple in-process key-value memory store for an AI agent.
    In production, replace with Redis, DynamoDB, or a vector DB.
    """

    def __init__(self):
        self._store: dict = {}
        self._history: list = []

    def remember(self, key: str, value: Any, category: str = "general") -> None:
        self._store[key] = {
            "value":      value,
            "category":   category,
            "timestamp":  datetime.datetime.now().isoformat()
        }

    def recall(self, key: str) -> Any:
        entry = self._store.get(key)
        return entry["value"] if entry else None

    def recall_all(self, category: str = None) -> dict:
        if category:
            return {k: v["value"] for k, v in self._store.items() if v["category"] == category}
        return {k: v["value"] for k, v in self._store.items()}

    def add_to_history(self, event: str) -> None:
        self._history.append({
            "event":     event,
            "timestamp": datetime.datetime.now().isoformat()
        })

    def context_summary(self) -> str:
        """Return a compact summary of memory for injection into the LLM context."""
        if not self._store and not self._history:
            return "No prior memory."
        parts = []
        if self._store:
            parts.append("Known facts:")
            for k, v in self._store.items():
                parts.append(f"  {k}: {v['value']}")
        if self._history:
            recent = self._history[-3:]
            parts.append("Recent actions:")
            for h in recent:
                parts.append(f"  {h['event']}")
        return "\n".join(parts)

    def __len__(self):
        return len(self._store)


class MemoryAgent(FunctionCallingAgent):
    """
    Extends FunctionCallingAgent with persistent external memory.
    Facts discovered during a session are stored and re-injected on subsequent calls.
    """

    MEMORY_EXTRACTION_PROMPT = """
Extract factual findings from this agent response that should be remembered for future queries.
Return a JSON array of objects: [{"key": "short_key", "value": "fact", "category": "category"}]
Categories: subscriber_data, network_status, financial, incident, preference
Only include concrete numbers and facts. If nothing is worth remembering, return [].

Response:
{response}
"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.memory = AgentMemory()

    def _extract_and_store_facts(self, response: str) -> None:
        try:
            raw = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": self.MEMORY_EXTRACTION_PROMPT.format(response=response)}],
                temperature=0.0,
                max_tokens=300,
                response_format={"type": "json_object"}
            ).choices[0].message.content
            parsed = json.loads(raw)
            # Handle both array and {"facts": [...]} forms
            facts = parsed if isinstance(parsed, list) else parsed.get("facts", [])
            for fact in facts:
                self.memory.remember(fact["key"], fact["value"], fact.get("category", "general"))
                if self.verbose:
                    print(f"  [MEMORY] Stored: {fact['key']} = {fact['value']}")
        except Exception as e:
            if self.verbose:
                print(f"  [MEMORY] Extraction failed: {e}")

    def run(self, user_request: str) -> str:
        # Inject memory context into the system prompt
        memory_ctx = self.memory.context_summary()
        request_with_memory = (
            f"{user_request}\n\n"
            f"[Prior memory context]\n{memory_ctx}"
        )

        response = super().run(request_with_memory)

        # Extract and store new facts from the response
        self._extract_and_store_facts(response)
        self.memory.add_to_history(f"Completed: {user_request[:60]}")

        return response


memory_agent = MemoryAgent(tools=TOOLS, tool_impl=TOOL_IMPL, verbose=True)
print("MemoryAgent ready.")

In [ ]:
# Session 1 — retrieve stats and let agent store them
print("--- Query 1: Initial data retrieval ---")
r1 = memory_agent.run(
    "Retrieve subscriber statistics for area KA-BLR (Bangalore) "
    "and summarise the key numbers."
)
print(f"\nMemory entries stored: {len(memory_agent.memory)}")

# Session 2 — agent can now use stored facts without calling the tool again
print("\n\n--- Query 2: Follow-up using memory ---")
r2 = memory_agent.run(
    "Using what you already know about Bangalore, calculate the revenue impact "
    "of churn at 550 rupees average monthly revenue. "
    "You do not need to retrieve stats again."
)

print("\n--- Memory store contents ---")
for k, v in memory_agent.memory.recall_all().items():
    print(f"  {k}: {v}")

## 5. Multi-Agent Collaboration — Concepts

A single agent operating within a single context window has practical limits: context size, specialisation depth, and fault isolation. Multi-agent systems address this by decomposing tasks across multiple specialised agents.

### Common Patterns

| Pattern | Description | Example |
|---|---|---|
| **Orchestrator-Worker** | One agent plans and delegates; sub-agents execute | Research agent delegates to search-agent and summariser-agent |
| **Pipeline** | Output of one agent is input to the next | Extractor -> Classifier -> Reporter |
| **Debate** | Two agents argue different positions; a judge evaluates | Risk assessment: bull agent vs bear agent |
| **Parallel** | Multiple agents run simultaneously on different sub-tasks | Analyse 5 regions in parallel |
| **Supervisor** | One agent monitors others and intervenes on errors | Quality-control agent reviewing outputs |

### Communication Between Agents

Agents communicate by passing structured messages. Options:
- **Direct function calls**: one agent calls another's `run()` method
- **Shared message queue**: agents post and consume from a broker (Kafka, Redis)
- **Shared state graph**: LangGraph's approach — a shared state object passed between nodes

We implement the orchestrator-worker pattern in full in Session 7 using LangGraph.

In [ ]:
# Minimal multi-agent demo: orchestrator calls two worker agents sequentially

def network_analyst_agent(node_id: str) -> str:
    """Worker: specialised in network status analysis."""
    result = noc_agent.run(
        f"Check node {node_id} status including history. "
        f"Return a 2-sentence status assessment only."
    )
    return result

def subscriber_analyst_agent(area_code: str, avg_revenue: float) -> str:
    """Worker: specialised in subscriber and revenue analysis."""
    result = noc_agent.run(
        f"Get subscriber stats for {area_code}. "
        f"Calculate churn revenue impact at {avg_revenue} rupees average monthly revenue. "
        f"Return two bullet points: subscriber risk and revenue risk."
    )
    return result

def orchestrator(node_id: str, area_code: str, avg_revenue: float) -> str:
    """
    Orchestrator: delegates to specialist workers,
    collects results, and synthesises a final report.
    """
    print("\n[ORCHESTRATOR] Delegating to network analyst...")
    network_report = network_analyst_agent(node_id)

    print("\n[ORCHESTRATOR] Delegating to subscriber analyst...")
    subscriber_report = subscriber_analyst_agent(area_code, avg_revenue)

    print("\n[ORCHESTRATOR] Synthesising final report...")
    synthesis_prompt = f"""
You are compiling a daily NOC operations briefing.
Combine the following two specialist reports into one structured briefing note.
Format: Title, Date, two sections (Network Status, Subscriber Risk), one Recommended Action.

Network Analysis:
{network_report}

Subscriber Analysis:
{subscriber_report}
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": synthesis_prompt}],
        temperature=0.2,
        max_tokens=400
    )
    return response.choices[0].message.content


noc_agent.verbose = False  # suppress sub-agent logs for clarity

final_briefing = orchestrator(
    node_id="NODE-TN-0042",
    area_code="TN-CHN",
    avg_revenue=450.0
)

print("\n" + "=" * 65)
print("  ORCHESTRATED DAILY BRIEFING")
print("=" * 65)
print(final_briefing)

## Session Summary

| Concept | Key Takeaway |
|---|---|
| Function calling | Native API feature that returns structured tool invocations — more reliable than prompt-parsing |
| Tool schema design | Descriptions must tell the model *when* to call the tool, not just what it does |
| ReAct pattern | Interleaved thought-action-observation traces make agents debuggable and more reliable |
| Memory types | In-context, key-value, and vector — each suited to different time horizons |
| Memory agent | Extract facts from responses, store them, re-inject on subsequent calls |
| Multi-agent | Orchestrator-worker and pipeline patterns scale beyond single-context limits |

## What Is Next

Session 7 introduces **LangGraph** — a graph-based framework for building stateful, multi-step agentic workflows. We formalise the patterns from Sessions 5 and 6 into reusable graph nodes with conditional routing and shared state.

---
*Agentic AI Nano Bootcamp | Day 2, Session 6*